In [0]:
-- =====================================================================
-- LBE Scene+ Redemption (>=1,500 pts, yesterday) -> Braze Format
-- Audience: CDE_IDs redeeming >=1,500 Scene+ points in one location-day
-- at Rec Room / Playdium yesterday, with an email on file.
-- Merges into data_exploration_hub.marketing_hub.custom_audiences_for_braze
-- =====================================================================

-- 1) Audience (QA-approved query)
CREATE OR REPLACE TEMP VIEW lbe_scene_redemption_audience AS
WITH CTE_A AS (
    SELECT
        t.CDE_ID,
        t.VisitDateId,
        t.LocationId,
        SUM(t.SCENE_Points_Redeemed) AS points_redeemed
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction AS t
    INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_location AS loc
        ON t.LocationId = loc.LocationID
    WHERE t.VisitDateId = CAST(date_format(current_date() - 1, 'yyyyMMdd') AS INT)
      AND t.CDE_ID IS NOT NULL
      AND t.CDE_ID > 0
      AND loc.LC_Location_Type_Description LIKE 'Restaurant%'   -- Rec Room + Playdium
    GROUP BY t.CDE_ID, t.VisitDateId, t.LocationId
    HAVING SUM(t.SCENE_Points_Redeemed) >= 1500
)
SELECT DISTINCT
    a.CDE_ID
FROM CTE_A AS a;


-- 2) Preview in Braze format (optional QA step)
SELECT
  braze.external_id,
  'cde_id' AS alias_label,
  CAST(src.CDE_ID AS STRING) AS alias_name,
  current_timestamp() AS updated_at,
  to_json(
    named_struct(
      'Campaign_names', array('LBE_Scene_Redemption_Nov_Dec_2026'),
      'LBE_Scene_Redemption_upload_date', date_format(current_date(), 'yyyy-MM-dd')
    )
  ) AS payload
FROM lbe_scene_redemption_audience src
INNER JOIN cpx_dataanalytics_gold.braze_dpp.braze_customer_profile braze
  ON CAST(src.CDE_ID AS STRING) = braze.alias_name;


-- -- 3) Merge into custom_audiences_for_braze -- ACTUAL TRANSMISSION TO BRAZE
-- CREATE TABLE IF NOT EXISTS data_exploration_hub.marketing_hub.custom_audiences_for_braze (
--   external_id STRING,
--   alias_label STRING,
--   alias_name STRING,
--   updated_at TIMESTAMP,
--   payload STRING
-- );

-- MERGE INTO data_exploration_hub.marketing_hub.custom_audiences_for_braze AS tgt
-- USING (
--   SELECT
--     braze.external_id,                            -- MANDATORY COLUMN
--     'cde_id' AS alias_label,                      -- MANDATORY COLUMN
--     CAST(src.CDE_ID AS STRING) AS alias_name,     -- MANDATORY COLUMN
--     current_timestamp() AS updated_at,            -- MANDATORY COLUMN
--     'LBE_Scene_Redemption_Nov_Dec_2026' AS campaign_name  -- MANDATORY COLUMN
--   FROM lbe_scene_redemption_audience src
--   INNER JOIN cpx_dataanalytics_gold.braze_dpp.braze_customer_profile braze
--     ON CAST(src.CDE_ID AS STRING) = braze.alias_name
-- ) AS stg
-- ON tgt.alias_name = stg.alias_name
-- WHEN MATCHED THEN
--   UPDATE SET
--     tgt.external_id = stg.external_id,
--     tgt.alias_label = stg.alias_label,
--     tgt.updated_at = stg.updated_at,
--     tgt.payload = to_json(
--       named_struct(  --- DO NOT CHANGE THE 'Campaign_names' Array --
--         'Campaign_names', array_union(
--           coalesce(
--             from_json(get_json_object(tgt.payload, '$.Campaign_names'), 'ARRAY<STRING>'),
--             from_json(get_json_object(tgt.payload, '$.Campaign_name'), 'ARRAY<STRING>'),
--             array()
--           ),
--           array(stg.campaign_name)
--         ),
--         -- CHANGE THE JSON STRUCT BASED ON CAMPAIGN HERE --
--         'LBE_Scene_Redemption_upload_date', date_format(current_date(), 'yyyy-MM-dd')
--       )
--     )
-- WHEN NOT MATCHED THEN
--   INSERT (external_id, alias_label, alias_name, updated_at, payload)
--   VALUES (
--     stg.external_id,
--     stg.alias_label,
--     stg.alias_name,
--     stg.updated_at,
--     to_json(
--       named_struct(  --- DO NOT CHANGE THE 'Campaign_names' Array --
--         'Campaign_names', array(stg.campaign_name),
--         -- CHANGE THE JSON STRUCT BASED ON CAMPAIGN HERE --
--         'LBE_Scene_Redemption_upload_date', date_format(current_date(), 'yyyy-MM-dd')
--       )
--     )
--   );